<a href="https://colab.research.google.com/github/samarae2508-design/week4_communityproject/blob/main/week4_assignment_communityproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics paho-mqtt

In [2]:
YOUR_NAME = "samra987"     # <- CHANGE THIS (example: Sathwik042)

BROKER = "broker.hivemq.com"  # backup brokers: broker.emqx.io, test.mosquitto.org

import re
assert (
    re.fullmatch(r"[a-z0-9]+", YOUR_NAME) and YOUR_NAME != "yourname123"
), "Change YOUR_NAME to your own first name + roll number (lowercase letters and numbers only)"

TOPIC = f"ssse/w4/{YOUR_NAME}/cmd"
print("Your topic is:")
print(TOPIC)
print("\nCopy this EXACT text into the TOPIC line of the Wokwi code.")

Your topic is:
ssse/w4/samra987/cmd

Copy this EXACT text into the TOPIC line of the Wokwi code.


In [9]:
#include <WiFi.h>
#include <PubSubClient.h>
#include <time.h>

// WiFi details
const char* WIFI_SSID = "Wokwi-GUEST";
const char* WIFI_PASS = "";

// MQTT details
const char* BROKER = "://hivemq.com";
const int MQTT_PORT = 1883;

WiFiClient espClient;
PubSubClient client(espClient);

// Buzzer pin
const int BUZZER_PIN = 18;

// Bell schedule (24-hour format)
struct BellSchedule {
  int hour;
  int minute;
  const char* event;
};

BellSchedule schedule[] = {
  {10, 30, "Morning Break"},
  {10, 45, "Break End"},
  {13, 0, "Lunch Start"},
  {13, 40, "Lunch End"},
  {16, 0, "College End"}
};

const int totalEvents = sizeof(schedule) / sizeof(schedule[0]);

int lastTriggeredDay[5] = {-1, -1, -1, -1, -1};

void connectWiFi() {
  WiFi.begin(WIFI_SSID, WIFI_PASS);

  while (WiFi.status() != WL_CONNECTED) {
    delay(500);
  }

  Serial.println("WiFi Connected");
}

void connectMQTT() {
  while (!client.connected()) {
    if (client.connect("ESP32CollegeBell")) {
      Serial.println("MQTT Connected");
    } else {
      delay(2000);
    }
  }
}

// Fixed function to generate audible sound waves and publish to your correct topic
void ringBell(const char* event) {
  Serial.print("Bell ringing: ");
  Serial.println(event);

  // Updated to match your exact assignment topic from the image
  if (client.connected()) {
    client.publish("ssse/w4/samra987/cmd", event);
  }

  // --- 2-BEEP BELL SEQUENCE ---
  // Beep 1
  tone(BUZZER_PIN, 1000); // 1 kHz sound frequency
  delay(1000);            // Keep sound on for 1 second
  noTone(BUZZER_PIN);     // Turn off sound
  delay(500);             // Silent pause for 0.5 seconds

  // Beep 2
  tone(BUZZER_PIN, 1000); // 1 kHz sound frequency
  delay(1000);            // Keep sound on for 1 second
  noTone(BUZZER_PIN);     // Turn off sound completely
}

void setup() {
  Serial.begin(115200);

  pinMode(BUZZER_PIN, OUTPUT);
  digitalWrite(BUZZER_PIN, LOW);

  connectWiFi();

  client.setServer(BROKER, MQTT_PORT);

  configTime(19800, 0,
             "pool.ntp.org",
             "://google.com");

  Serial.println("Automatic College Bell System Started");
}

void loop() {
  if (!client.connected()) {
    connectMQTT();
  }

  client.loop();

  struct tm timeInfo;

  if (!getLocalTime(&timeInfo, 1000)) {
    Serial.println("Waiting for time...");
    return;
  }

  for (int i = 0; i < totalEvents; i++) {

    if (timeInfo.tm_hour == schedule[i].hour &&
        timeInfo.tm_min == schedule[i].minute &&
        lastTriggeredDay[i] != timeInfo.tm_yday) {

      lastTriggeredDay[i] = timeInfo.tm_yday;

      ringBell(schedule[i].event);
    }
  }

  delay(1000);
}


SyntaxError: invalid syntax (3495858771.py, line 5)

In [4]:
import time, cv2, numpy as np
from base64 import b64decode
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js

def start_camera():
    display(Javascript('''
      async function startCam() {
        const video = document.createElement('video');
        video.style.width = '240px';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(video);
        video.srcObject = stream;
        await video.play();
        window.cam = {video: video, stream: stream, canvas: document.createElement('canvas')};
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      }
    '''))
    eval_js('startCam()')

def get_frame(quality=0.6):
    data = eval_js('''
      (function() {
        const c = window.cam;
        c.canvas.width = c.video.videoWidth;
        c.canvas.height = c.video.videoHeight;
        c.canvas.getContext('2d').drawImage(c.video, 0, 0);
        return c.canvas.toDataURL('image/jpeg', %s);
      })()
    ''' % quality)
    jpg = b64decode(data.split(',')[1])
    return cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)

def stop_camera():
    eval_js('window.cam.stream.getTracks().forEach(t => t.stop())')

print("Camera helpers ready")

Camera helpers ready


In [8]:
from ultralytics import YOLO
import sys

model = YOLO("yolov8n.pt")        # downloads ~6 MB the first time (if it fails, use "yolo11n.pt")

try:
    start_camera()
    time.sleep(1)                     # the very first frame is often black
    frame = get_frame()
    result = model(frame, verbose=False, conf=0.4)[0]
    print("I can see:", [result.names[int(c)] for c in result.boxes.cls])
    display(Image(data=cv2.imencode(".jpg", result.plot())[1].tobytes()))
    stop_camera()
except Exception as e:
    if "Permission denied" in str(e) or "NotAllowedError" in str(e):
        print("\n[ERROR] Camera Permission Denied!")
        print("Please look at your browser's address bar, click the camera icon, allow camera permissions for Google Colab, and run this cell again.")
    else:
        raise e

<IPython.core.display.Javascript object>

KeyboardInterrupt: 